In [24]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, WebSearchTool, function_tool, trace

dotenv.load_dotenv()

True

In [25]:
# Setup ChromaDB client and collection
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [26]:
# Define the calorie lookup tool

@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [27]:
# Define the agent with the calorie lookup tool and web search tool
calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if it's about a meal, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use web search to find the exact recipe and ingredients.
    * Once you know the ingredients, always use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 8 times.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()],
)

In [29]:
with trace("Nutrition Assistant with Web Search"):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A typical full English breakfast is roughly 800–1,200 kcal per serving, depending on portions and cooking fats. Here’s a common breakdown for a hearty plate (approximate calories per item):

- Eggs (2 large): ~95 kcal
- Bacon (2 slices): ~160 kcal
- Sausage (2 links): ~400 kcal
- Baked beans (1/2 cup): ~120 kcal
- Mushrooms (1 cup cooked): ~30–35 kcal
- Tomato (1 medium): ~20 kcal
- Toast (2 slices, white bread) with butter: ~260–300 kcal
- Butter (for cooking or spreading): ~90–100 kcal

 Estimated total: about 1,100–1,200 kcal (heavily depends on sausage type, portions, and added fats). If you use fewer sausages, leaner bacon, or less butter, the total can drop toward ~800–900 kcal.

Sources note that typical full English breakfasts vary widely in calories, often cited around 800–1,600 kcal depending on serving size and fats used. 


In [ ]:
#Question: At which stage did you find the Web Search Tool use in the Traces UI?
#Answer: I found the Web Search Tool being used at stage 1, where the agent searched the web to figure out the exact ingredients of the meal.